# 28 · 时间外验证：只考模型没读过的病例

## 这一章在整个项目里的位置

```
00 总览 → 01 存档旧项目 ✅ → 02 新项目骨架 ✅ → 03 自动检查 ✅ → 04 GitHub 与 CI ✅ → 05 准备考题 ✅ → 06 评测集 ✅ → 07 判分 ✅ → 08 MLflow 评测 ✅ → 09 第一个基线 ✅ → 10 第一个工具 ✅ → 11 第二个工具 ✅ → 12 第三个工具 ✅ → 13 标准接口 MCP ✅ → 14 固定流程 ✅ → 15 第一个 agent ✅ → 16 看错题 ✅ → 17 找相似病人 ✅ → 18 疾病介绍（RAG） ✅ → 19 基因工具 ✅ → 20 大考 ✅ → 21 正式评测 ✅ → 22 做成服务 ✅ → 23 和别人比 ✅ → 24 和大模型比 ✅ → 25 和 DeepRare 比 ✅ → 26 读病历 ✅ → 27 交付 ✅ → [28 时间外验证 ✅] → 29 外部验证
```

> **这一章的代码在分支 `feat/temporal-external` 里**（和第 29 章同一个合并请求）。新考卷 47 道题，13 次考试；DeepSeek 约 ¥5，OpenRouter 约 $1。**运行这本笔记不花钱**：考试的命令只列出来不重跑，结果从 MLflow 和逐题表里读。
>
> **这一章分两段写。** 第 1–4 步是考之前写的：新病例从哪来、论文哪天公开的、怎么选题、方案。第 5 步起是考完以后填的。

## 背景：留出集还剩一样东西遮不住

第 21 章的留出集已经很干净：按论文分开，只考一次，每道题遮住病人自己的论文，考完自动审计，泄露 0 次。

但 [`docs/leakage.md`](../leakage.md) 第六节写过一条**工具层面遮不住**的：**大模型自己的记忆**。DeepSeek 训练时很可能读过这些论文。举那里的真例子：有一道题的答案是真性小眼球 2 型，HPO 里关于它的资料只来自这道题自己的论文，遮挡以后对工具来说它不存在；可是"大模型直接猜"（什么工具都不给）把它排在了第 2 名。是记得这篇论文，还是从"眼球小"顺着病名猜出来的？分不清。

第 21 章按发表年份拆开看过一次：

| 发表年份 | 题数 | 注定答不对 | 大模型直接猜 | 传统方法（不用模型） | 最终 agent |
|---|---|---|---|---|---|
| 2010 年及以前 | 158 | 7% | 27.2% | 26.5% | 62.6% |
| 2024 年以后 | 107 | 49% | 7.3% | 7.3% | 32.7% |

（能答的题上的第 1 名）越新的论文分数越低，但**不用模型的传统方法掉得一样多**，所以分不清是"记忆没了"还是"新病本来就难"。而且年份只精确到年，模型的训练截止却在年中。

**打个比方**：卷子没泄露，但学生可能在课外书上见过原题。只有考**今年新出的题**，才能排除这一条。

**公司里的做法**：这叫**时间外验证**（temporal validation）：按时间切，只用某一天以后才出现的数据考。大模型评测里专门有一类"防污染"的评测集（contamination-free benchmark，例如 LiveBench、LiveCodeBench），定期换上新出的题，原因一样：模型训练时可能见过旧题。

## 这一章做的 7 件事

| # | 做什么 | 用的技术 | 解决什么问题 |
|---|---|---|---|
| 1 | 拿到更新的病例 | 固定在 GitHub 的一次提交；读取程序认两种目录结构 | Phenopacket Store 还没发新版本 |
| 2 | 查每篇论文哪天公开的 | PubMed 官方接口 E-utilities；日期表进 git | 知道哪些论文在模型训练截止以后 |
| 3 | 选题 | 写死的规则 + 自动检查 | 47 道模型没读过、系统也没碰过的题 |
| 4 | 先写方案 | 预注册：`docs/temporal_plan.md` | 不看了结果再挑 |
| 5 | 开考 | 13 次考试，`--final`，泄露审计 | 只考一次 |
| 6 | 结果 | 置信区间、逐题配对 | 回答方案里的问题 |
| 7 | 模型记忆占了多少分 | 用不用模型的系统当参照 | 分清"记忆没了"和"题变难了" |

---

## 准备

运行下面这一格。它定义一个叫 `run` 的小工具，用来执行命令并显示结果。**不用看代码。**

> 运行 `uv` 的命令时，有时会先出现几行 `Building` / `Installed`：那是 uv 发现代码有变化，顺手更新了一下环境，可以忽略。

In [ ]:
import os, subprocess
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
os.environ.update(NO_COLOR="1", GH_PAGER="cat", GIT_PAGER="cat")
for d in ("/opt/homebrew/bin", "/usr/local/bin", str(Path.home() / ".local/bin")):  # 编辑器不是从终端打开时，也找得到 uv
    if Path(d).is_dir() and d not in os.environ["PATH"].split(os.pathsep):
        os.environ["PATH"] += os.pathsep + d
import logging, warnings
warnings.filterwarnings("ignore", message="IProgress not found")   # 一条和我们无关的提示
logging.getLogger("mlflow").setLevel(logging.WARNING)
def run(cmd, check=True):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode != 0:
        raise RuntimeError(f"failed: {cmd}")
print("项目目录:", ROOT)

---

## 第 1 步 · 拿到更新的病例

### 在干什么
从 Phenopacket Store 拿到 0.1.27 以后新整理的病人。

### 背景
项目一直用 Phenopacket Store 0.1.27（2026 年 6 月发布）。之后几个月，整理的人又加了不少病人，但还没发新版本，只在 GitHub 的主线上。查了一下：10 月 7 日那次提交，比 0.1.27 多了 **403 个病人，来自 102 篇 0.1.27 里没有的论文**。这些病人，我们的系统从来没见过。

### 用了什么技术，为什么

| 做法 | 为什么 |
|---|---|
| **固定在一次提交上**（`85ffd78`），和其他数据一样登记在 `evals/datasets.toml`，带校验码 | 主线每天都在变，"主线上的数据"今天和明天不是同一份。固定到一次提交，就和固定一个版本一样：谁来下载都是同一份。第 26 章的 PhenoBERT 也是这么固定的 |
| 读取程序认两种目录结构 | 正式版本里是 `版本/病人组/文件`；GitHub 打包里是 `notebooks/病人组/phenopackets/文件`，旁边还有整理用的表格（也是 JSON 文件），不能当成病人读进来 |
| 0.1.27 里没有的论文，病人标成 `external` | 只用来考：永远不进开发，不进"找相似病人"的病例库 |

### 公司里什么情况怎么做
- **上游没发版本，又必须用新数据**：固定到一次提交的编号（commit hash），而不是写"最新"。不然半年后谁也说不清当时用的是哪份
- **改了读数据的代码**：先证明旧数据读出来一模一样，再拿它读新数据。这叫**回归检查**。这里改完读取程序，先重读 0.1.27，和以前存下的 10,374 个病人逐个比，完全一样才往下走；现在有测试专门守着

### 执行
下载（已经下载过的只核对校验码），再读成统一的病例格式：

In [ ]:
run("uv run raredx data fetch phenopacket-store-snapshot")
run("uv run raredx data build phenopacket-store-snapshot")

from collections import Counter
from raredx.data import cases as store
snapshot = store.read(ROOT / "data/processed/phenopacket-store-snapshot/85ffd78")
release = store.read(ROOT / "data/processed/phenopacket-store/0.1.27")
print("主线快照：", Counter(c.split for c in snapshot))
print("0.1.27：  ", Counter(c.split for c in release))
new = [c for c in snapshot if c.split == "external"]
print(f"新病人 {len(new)} 个，来自 {len({c.pmid for c in new})} 篇论文，{len({c.disease_id for c in new})} 种病")

### 结果怎么看
- 主线快照 10,776 个病人：`design` 6,013、`heldout` 4,360，和 0.1.27 的两半一样（开发那一半少了 1 个：整理的人后来删掉了一个病人）；多出来的 **403 个标成了 `external`**
- 新病人来自 102 篇论文、49 种病
- 开发、病例库照旧只用 0.1.27。快照只提供新题

---

## 第 2 步 · 查每篇论文哪天公开的

### 在干什么
给每篇候选论文（留出集 840 篇 + 新的 102 篇）查出"第一次能被人读到"的日子。

### 背景
一篇论文有好几个日子。看一个真例子，PMID:40996075：

| 日子 | 日期 |
|---|---|
| 网络首发（epub） | 2025-09-25 |
| 进 PubMed（entrez） | 2025-09-25 |
| 期刊出版（issue） | 2026 年 2 月 |

模型训练时抓网页，网络首发那天就可能抓到。所以要的是**最早公开的那一天**，不是期刊上印的日子。

### 用了什么技术，为什么

| 做法 | 为什么 |
|---|---|
| **NCBI E-utilities**（PubMed 的官方接口，`esummary`） | 生物信息领域查文献信息的标准做法；免费、不用账号；一次能问 200 篇。不去抓网页 |
| 取三个日子里**最早**的；只有月份的按 1 号，只有年份的按 1 月 1 号 | **宁早不晚**：把一篇论文算早了，最多是少一道题；算晚了，就可能把模型读过的论文当成没读过 |
| 结果存成 `evals/pubmed_dates.tsv`，提交进 git；只查表里还没有的 | 选题规则依赖这张表，表进 git，谁来重建考卷都一样，断网也行 |

> **一个真实踩过的坑**：第一版只取"进 PubMed 的日子"和"网络首发日"。拿第 21 章的年份表一核对，有 **11 篇论文反而晚了一年**。原因：老论文常常先在纸质期刊上出版，过了一阵才进 PubMed。比如 PMID:10874631，**1999 年 6 月**出版，**2000 年 6 月 30 日**才进 PubMed。把期刊出版日也算进来，重查以后，没有一篇比第 21 章晚。
>
> 反过来也有：PMID:40497796，**2025 年 6 月 11 日**就进了 PubMed，期刊出版日却写着 2026 年 2 月。只看出版日，它会被当成 2026 年的论文。

### 公司里什么情况怎么做
- **按时间切的评测，"时间"的定义要写死**：用哪个日子、缺了怎么办，写进代码和方案，不然换个人算出来是另一份题
- **外部接口的结果存下来**：接口会变、会限流、会下线；存进仓库的表才是评测的一部分

### 执行
表已经查好了，再运行一次只会说"0 篇要查"：

In [ ]:
run("uv run raredx data pubmed-dates")
import csv
rows = {r["pmid"]: r for r in csv.DictReader(open(ROOT / "evals/pubmed_dates.tsv"), delimiter="\t")}
for pmid in ("PMID:40996075", "PMID:10874631", "PMID:40497796"):
    r = rows[pmid]
    print(f"{pmid}  首次公开 {r['first_public']}   进 PubMed {r['entrez']}   网络首发 {r['epub'] or '—':<10}   期刊出版 {r['issue']}")

### 结果怎么看
- `942 of 942 papers dated`：每篇候选论文都查到了日子
- 三个例子的"首次公开"都取了最早的那个：2025-09-25、1999-06-01、2025-06-11

---

## 第 3 步 · 选题：模型训练截止以后的论文

### 在干什么
选出这一章的考卷 `evals/splits/temporal-2026-10.tsv`。

### 背景
截止日定在 **2025-06-01**。我们所有系统里的大模型都是 deepseek-flash，第三方资料普遍写它的训练数据截止在 2025 年 5 月。DeepSeek 自己没有正式公布，这一点照实写进了方案。

### 用了什么技术，为什么
规则写在代码里（`src/raredx/evals/splits.py` 的 `temporal`），三条：

1. **论文首次公开在 2025-06-01 当天或以后**
2. **系统没被它开发过**，两个来源：
   - 留出集的论文：用第 21 章考过的**同一道题**
   - 0.1.27 以后新加的论文：每篇按固定种子抽一道（和 dev-200、留出集一样，一篇论文一道，免得一家人被数很多遍）
3. **开发集的论文一篇不要**：开发时看过它们的分数，而且它们的病人就在"找相似病人"的病例库里

考卷和以前的一样**冻结**，并且有自动检查：

| 检查 | 什么时候跑 |
|---|---|
| 没有开发集的论文；一篇论文恰好一道；留出集论文用的是第 21 章那道题；每篇论文都在截止日以后 | 每次提交代码（CI 里，不需要数据） |
| 用数据和规则重新选一遍，和考卷一字不差 | 本机，`raredx eval check` |

### 公司里什么情况怎么做
- **时间外的测试集，要排除所有"开发时碰过"的来源**，不只是"日期够新"。这里开发集的论文就算够新也不要
- **新数据会越来越多**：这份考卷的名字带日期（`2026-10`）。Phenopacket Store 出新版本以后，同样的规则会选出更多题，那是一份**新的**考卷，旧的不改

### 执行

In [ ]:
run("uv run raredx eval splits --external")
run("uv run raredx eval check")

from raredx.evals import splits
from raredx.evals.hits import answerable
from raredx.knowledge import hpo
knowledge = hpo.load(ROOT / "data/raw/hpo/v2026-09-01")
by_id = {c.id: c for c in snapshot}
exam = [by_id[case_id] for case_id, _ in splits.read(ROOT / "evals/splits/temporal-2026-10.tsv")]
heldout_papers = {p for _, p in splits.read(ROOT / "evals/splits/heldout-papers.tsv")}
dates = sorted(rows[c.pmid]["first_public"] for c in exam)
print(f"{len(exam)} 道题，{len({c.disease_id for c in exam})} 种病；论文日期 {dates[0]} 到 {dates[-1]}")
print(f"来自留出集 {sum(c.pmid in heldout_papers for c in exam)} 篇，新论文 {sum(c.pmid not in heldout_papers for c in exam)} 篇")
print(f"注定答不对：{sum(not answerable(c, knowledge) for c in exam)} 道")

### 结果怎么看
- 四份新考卷都是 `unchanged`：已经冻结了，再运行只确认没变（另外三份是第 29 章的）
- **47 道题，44 种病**；论文从 2025-06-03 到 2026-07-19；21 篇来自留出集，26 篇是新论文；47 道都有致病基因
- **14 道注定答不对**：13 道的病只在自己那篇论文里描述过（遮挡以后不存在），1 道的病 HPO 没收录。能答的 33 道
- **题很少**：47 道的误差范围约正负 14 个百分点。只看得出大的差别

---

## 第 4 步 · 先写方案

### 在干什么
任何系统碰这 47 道题之前，写好方案提交：[`docs/temporal_plan.md`](../temporal_plan.md)。和第 21 章一样，考完只在最后一节填数字。

### 背景
47 道题很少，方案里最要紧的是**事先说清能测出什么、测不出什么**。

### 主要结论（只有一个）

> **在模型不可能读过的病例上，最终 agent 的第 1 名命中率，是否高于同一个模型直接猜。**

为什么选这一条，用大白话说：
- "直接猜"只能靠模型脑子里的东西。在它没读过的论文上，靠记忆答对的路断了，只剩从症状推理
- agent 用的是**同一个模型**，但答案来自工具查到的资料
- 如果 agent 在这里仍然明显胜过直接猜，说明它的本事来自工具，不是来自模型记得论文

**47 道够不够**：第 21 章 agent 对直接猜是 257 比 21，约三分之一的题两者不一样。照这个比例，这里大约 15 道两者不一样，测得出来。第 21 章的主要结论（agent 胜过固定流程）只有 8% 的题两者不一样，这里大约 4 道，**测不出来**，所以它只当次要结论，事先写明"很可能不显著"。

### 两件要事先说明的事
1. **47 道里有 21 道第 21 章考过**，各系统答没答对已经记在逐题表里。写方案时我没有去看这 21 道的逐题结果，比较对象全部照第 21–25 章定
2. **三家通用大模型的训练截止更晚**：GPT-6.1 是 2026 年 4 月 30 日（第三方转述 OpenAI 的说法），Claude Sonnet 5.5 是 2026 年 6 月（Anthropic 官方），Gemini 4 没有公布。这份题只对 deepseek-flash 是干净的；对它们，分数仍是偏高的上限，比较对 agent 不利

### 公司里什么情况怎么做
- **小样本的评测，事先算一下"大概有几道题会不一样"**：配对比较能不能显著，看的是两个系统答得不一样的题有几道，不是总题数
- **测不出来的结论也写进方案**，标明"预计不显著"。考完不显著，就照实报，不换成别的说法

### 面试怎么说
> "The held-out set can't hide the model's memory of the papers, so I built a temporal test: every case from a paper first public after deepseek-flash's training cutoff that no system was developed on — 47 cases, dated through PubMed's E-utilities. With so few cases the plan says up front what it can detect: the primary claim is the agent over the same model asked directly, where about a third of cases disagree; the held-out claim against the fixed pipeline is a secondary, expected to be underpowered." 

---

**第 5 步起（开考、结果、模型记忆占多少分）考完以后补上。**